# Notebook 05 — Output Visualization

**Purpose:** Generate final charts and the summary table. This provides a clear visual narrative of the risk-return trade-offs for the treasury manager.

**Inputs:** 
- `data/processed/frontier_portfolios.csv`
- `data/processed/named_portfolios.json`

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import json
import os

os.makedirs('../data/processed/charts', exist_ok=True)

frontier_df = pd.read_csv('../data/processed/frontier_portfolios.csv')
with open('../data/processed/named_portfolios.json') as f:
    named = json.load(f)

## 1. Efficient Frontier

Plot Yield vs CVaR.

In [2]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=frontier_df['cvar_95'],
    y=frontier_df['expected_return'],
    mode='lines+markers',
    name='Efficient Frontier (CVaR)',
    marker=dict(size=4)
))

# Add named portfolios
for name in ['min_risk', 'balanced', 'max_yield']:
    p = named[name]
    fig.add_trace(go.Scatter(
        x=[p['cvar_95']],
        y=[p['expected_return']],
        mode='markers+text',
        name=name.capitalize(),
        text=[name.capitalize()],
        textposition="top center",
        marker=dict(size=12, symbol='star')
    ))

fig.update_layout(
    title='Efficient Frontier: Annualized Yield vs 95% CVaR',
    xaxis_title='95% CVaR (Expected Tail Loss)',
    yaxis_title='Annualized Risk-Adjusted Yield (IDR)',
    template='plotly_white'
)
fig.write_image('../data/processed/charts/efficient_frontier.png')
fig.show()

## 2. Allocation Stack

How the portfolio composition changes as we target higher yields.

In [3]:
w_cols = [c for c in frontier_df.columns if c.startswith('w_')]
frontier_df['total'] = frontier_df[w_cols].sum(axis=1)

fig2 = px.area(
    frontier_df,
    x=np.arange(len(frontier_df)),
    y=w_cols,
    title='Portfolio Allocation vs Portfolio Index (Risk $\rightarrow$ Yield)',
    labels={'x': 'Portfolio Index', 'value': 'Weight'}
)
fig2.update_layout(template='plotly_white')
fig2.write_image('../data/processed/charts/allocation_stack.png')
fig2.show()

## 3. Summary for Dashboard

In [4]:
summary_rows = []
for name, p in named.items():
    row = {
        'portfolio': name,
        'yield': p['expected_return'],
        'cvar_95': p['cvar_95']
    }
    # Append weights
    for k, v in p.items():
        if k.startswith('w_'):
            row[k.replace('w_', '')] = v
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('../data/processed/final_summary.csv', index=False)
summary_df

,portfolio,yield,cvar_95,aave,mmf,pendle_pt,pendle_yt,sbn
0,min_risk,0.123641,-0.083134,0.304477,8.488832e-07,0.500001,0.150001,0.045520
1,balanced,0.123643,-0.083135,0.304478,5.223269e-06,0.500008,0.150010,0.045499
2,max_yield,0.124629,-0.082888,0.344897,-3.313167e-08,0.500000,0.150000,0.005103
